# 03 - Forecasting Models
- **Q1:** Sales revenue forecast Q2/2026 (Prophet + LightGBM)
- **Q2:** Color demand forecast
- **Q3:** Dealer activity forecast (XGBoost + RFM)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA = Path('../data/processed')
FCAST = DATA / 'forecasts'
FCAST.mkdir(exist_ok=True)

print('Imports OK')

Imports OK


## Q1 - Dự báo doanh số Q2/2026

### Q1.A - Prophet baseline

In [2]:
from prophet import Prophet
import sys
sys.path.insert(0, str(Path('../dashboard').resolve()))
from time_series_utils import weekly_revenue_observed

fs = pd.read_parquet(DATA / 'fact_sales_clean.parquet')
fs['order_date'] = pd.to_datetime(fs['order_date'])

# Only weeks with actual transactions — missing calendar months are not zero revenue
weekly = weekly_revenue_observed(fs)

train_end = pd.Timestamp('2025-12-31')
train_p = weekly[weekly['ds'] <= train_end].copy()
val_p = weekly[weekly['ds'] > train_end].copy()

print(f'Prophet train: {len(train_p)} weeks, val: {len(val_p)} weeks')
print(f'  (observed weeks only; no gap-filled zeros)')

m = Prophet(
    yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
    seasonality_mode='additive'
)
m.fit(train_p)

val_pred = m.predict(val_p[['ds']])
val_pred['yhat'] = val_pred['yhat'].clip(lower=0)
val_mape = mean_absolute_percentage_error(val_p['y'], val_pred['yhat'])
val_rmse = np.sqrt(mean_squared_error(val_p['y'], val_pred['yhat']))
print(f'Prophet validation - MAPE: {val_mape:.4f}, RMSE: {val_rmse:,.0f}')

07:40:01 - cmdstanpy - INFO - Chain [1] start processing


Prophet train: 52, val: 14


07:40:02 - cmdstanpy - INFO - Chain [1] done processing


Prophet validation - MAPE: 5.5156, RMSE: 8,140,071,404


In [3]:
m_full = Prophet(
    yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
    seasonality_mode='additive'
)
m_full.fit(weekly)

future_weeks = m_full.make_future_dataframe(periods=13, freq='W-MON')
forecast_prophet = m_full.predict(future_weeks)
q2_prophet = forecast_prophet[forecast_prophet['ds'] >= '2026-04-01'][['ds','yhat','yhat_lower','yhat_upper']].copy()
q2_prophet.columns = ['week','revenue_prophet','lower','upper']
for col in ['revenue_prophet', 'lower', 'upper']:
    q2_prophet[col] = q2_prophet[col].clip(lower=0)
q2_prophet['month'] = q2_prophet['week'].dt.month
print('Prophet Q2 forecast (weekly):')
print(q2_prophet)
print(f'\nProphet Q2 total: {q2_prophet["revenue_prophet"].sum():,.0f} VND')

07:40:02 - cmdstanpy - INFO - Chain [1] start processing


07:40:02 - cmdstanpy - INFO - Chain [1] done processing


Prophet Q2 forecast (weekly):
         week  revenue_prophet         lower         upper  month
65 2026-04-06     3.799364e+09  1.657814e+09  5.960358e+09      4
66 2026-04-13     7.052169e+08 -1.421908e+09  2.882972e+09      4
67 2026-04-20    -1.142888e+09 -3.218113e+09  9.797260e+08      4
68 2026-04-27    -1.396241e+09 -3.488638e+09  7.870611e+08      4
69 2026-05-04    -4.998344e+08 -2.625861e+09  1.490683e+09      5
70 2026-05-11     5.162818e+08 -1.479267e+09  2.638274e+09      5
71 2026-05-18     7.611294e+08 -1.399559e+09  2.842964e+09      5
72 2026-05-25     1.885913e+08 -1.842955e+09  2.284091e+09      5
73 2026-06-01    -4.540968e+08 -2.648461e+09  1.590119e+09      6
74 2026-06-08    -4.622219e+08 -2.533667e+09  1.625219e+09      6
75 2026-06-15     8.048160e+07 -2.115603e+09  2.090669e+09      6
76 2026-06-22     4.728396e+08 -1.707952e+09  2.396589e+09      6
77 2026-06-29     2.619701e+08 -1.876932e+09  2.197917e+09      6
78 2026-07-06    -2.382734e+08 -2.289182e+09  

In [4]:
groups = fs['group_code'].dropna().unique()
group_forecasts = []

for g in groups:
    gdf = weekly_revenue_observed(fs[fs['group_code'] == g])
    if len(gdf) < 10:
        continue
    mg = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False, seasonality_mode='additive')
    mg.fit(gdf)
    fut = mg.make_future_dataframe(periods=13, freq='W-MON')
    pred = mg.predict(fut)
    q2g = pred[pred['ds'] >= '2026-04-01'][['ds','yhat']].copy()
    q2g['yhat'] = q2g['yhat'].clip(lower=0)
    q2g['group_code'] = g
    group_forecasts.append(q2g)

group_fc = pd.concat(group_forecasts, ignore_index=True)
group_fc.columns = ['week','revenue_prophet','group_code']

group_monthly = group_fc.copy()
group_monthly['month'] = group_monthly['week'].dt.month
gm_summary = group_monthly.groupby(['group_code','month'])['revenue_prophet'].sum().reset_index()
print('Prophet forecast by group x month:')
print(gm_summary.pivot(index='group_code', columns='month', values='revenue_prophet').fillna(0))

07:40:02 - cmdstanpy - INFO - Chain [1] start processing


07:40:02 - cmdstanpy - INFO - Chain [1] done processing


07:40:02 - cmdstanpy - INFO - Chain [1] start processing


07:40:02 - cmdstanpy - INFO - Chain [1] done processing


07:40:03 - cmdstanpy - INFO - Chain [1] start processing


...

07:40:03 - cmdstanpy - INFO - Chain [1] start processing


07:40:03 - cmdstanpy - INFO - Chain [1] done processing


07:40:03 - cmdstanpy - INFO - Chain [1] start processing


07:40:03 - cmdstanpy - INFO - Chain [1] done processing


Prophet forecast by group x month:
month                   4             5             6             7
group_code                                                         
CITYBIKE_P   1.270736e+09  6.562910e+08 -5.769640e+07 -1.971515e+08
KIDBIKE_1    2.682883e+08  1.299578e+08 -3.159684e+06 -4.590085e+07
KIDBIKE_2    3.619391e+07  4.610830e+07  1.683336e+06 -1.478325e+07
SPORTBIKE_A  6.713188e+07 -3.349608e+07 -4.672925e+08  0.000000e+00
SPORTBIKE_S  8.284951e+07  2.217848e+07  3.859399e+05 -6.379252e+06


### Q1.B - LightGBM

In [5]:
pw = pd.read_parquet(DATA / 'product_week_panel.parquet')
pw['week_start'] = pd.to_datetime(pw['week_start'])

feature_cols = [
    'month','week_of_year','quarter','group_code_enc','color_enc',
    'unit_price','qty_lag_1w','qty_lag_2w','qty_lag_4w',
    'rev_lag_1w','rev_lag_2w','rev_lag_4w',
    'qty_roll_4w','qty_roll_8w','rev_roll_4w','rev_roll_8w',
    'weeks_since_first','cum_quantity','n_price_changes'
]

pw_model = pw.dropna(subset=feature_cols).copy()

train_lgb = pw_model[pw_model['week_start'] <= '2025-12-31']
val_lgb = pw_model[(pw_model['week_start'] > '2025-12-31') & (pw_model['week_start'] <= '2026-03-31')]

print(f'LightGBM train: {len(train_lgb)}, val: {len(val_lgb)}')

X_train = train_lgb[feature_cols]
y_train = train_lgb['quantity']
X_val = val_lgb[feature_cols]
y_val = val_lgb['quantity']

model_lgb = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=8,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=10, random_state=42, verbose=-1
)
model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

val_pred_lgb = model_lgb.predict(X_val).clip(min=0)

mask = y_val > 0
lgb_mape = mean_absolute_percentage_error(y_val[mask], val_pred_lgb[mask])
lgb_rmse = np.sqrt(mean_squared_error(y_val, val_pred_lgb))
lgb_mae = mean_absolute_error(y_val, val_pred_lgb)
print(f'LightGBM validation - MAPE: {lgb_mape:.4f}, RMSE: {lgb_rmse:.2f}, MAE: {lgb_mae:.2f}')

LightGBM train: 12054, val: 3198


LightGBM validation - MAPE: 1.2164, RMSE: 28.58, MAE: 12.13


In [6]:
model_full = lgb.LGBMRegressor(
    n_estimators=model_lgb.best_iteration_ if hasattr(model_lgb, 'best_iteration_') else 300,
    learning_rate=0.05, max_depth=8, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
    random_state=42, verbose=-1
)
train_full = pw_model[pw_model['week_start'] <= '2026-03-31']
model_full.fit(train_full[feature_cols], train_full['quantity'])

last_week = pw_model[pw_model['week_start'] == pw_model['week_start'].max()].copy()
q2_weeks = pd.date_range('2026-04-06', periods=13, freq='W-MON')

all_preds = []
rolling_data = pw_model.copy()

for wk in q2_weeks:
    pred_rows = last_week.copy()
    pred_rows['week_start'] = wk
    pred_rows['month'] = wk.month
    pred_rows['week_of_year'] = wk.isocalendar()[1]
    pred_rows['quarter'] = (wk.month - 1) // 3 + 1
    pred_rows['weeks_since_first'] = pred_rows['weeks_since_first'] + 1

    X_pred = pred_rows[feature_cols]
    pred_qty = model_full.predict(X_pred).clip(min=0)
    pred_rows['quantity'] = pred_qty
    pred_rows['revenue'] = pred_qty * pred_rows['unit_price']
    all_preds.append(pred_rows[['product_code','week_start','quantity','revenue','unit_price',
                                'group_code_enc','color_enc','product_name','group_code','color_std','line_name']])

    for lag_col, src_col in [('qty_lag_1w','quantity'),('rev_lag_1w','revenue')]:
        last_week[lag_col] = pred_rows[src_col].values
    last_week['qty_lag_2w'] = last_week['qty_lag_1w']
    last_week['rev_lag_2w'] = last_week['rev_lag_1w']
    last_week['cum_quantity'] = last_week['cum_quantity'] + pred_qty
    last_week = pred_rows.copy()
    for c in feature_cols:
        if c not in last_week.columns:
            last_week[c] = train_full.groupby('product_code')[c].last().reindex(last_week['product_code']).values

q2_preds = pd.concat(all_preds, ignore_index=True)
q2_preds['month'] = q2_preds['week_start'].dt.month

print(f'LightGBM Q2 predictions: {q2_preds.shape}')
print(f'Total predicted revenue Q2: {q2_preds["revenue"].sum():,.0f} VND')
print(f'Total predicted quantity Q2: {q2_preds["quantity"].sum():,.0f}')

LightGBM Q2 predictions: (3198, 12)
Total predicted revenue Q2: 26,421,507,842 VND
Total predicted quantity Q2: 15,420


In [7]:
monthly_rev = q2_preds.groupby('month').agg(
    revenue=('revenue','sum'), quantity=('quantity','sum')
).reset_index()
monthly_rev['month_name'] = monthly_rev['month'].map({4:'Tháng 4',5:'Tháng 5',6:'Tháng 6'})
print('=== Doanh thu dự báo theo tháng ===')
print(monthly_rev)

group_rev = q2_preds.groupby(['group_code','month']).agg(
    revenue=('revenue','sum'), quantity=('quantity','sum')
).reset_index()
print('\n=== Doanh thu theo nhóm SP x tháng ===')
print(group_rev.pivot(index='group_code', columns='month', values='revenue').fillna(0))

=== Doanh thu dự báo theo tháng ===
   month       revenue     quantity month_name
0      4  8.132969e+09  4746.454345    Tháng 4
1      5  8.128239e+09  4743.966296    Tháng 5
2      6  1.016030e+10  5929.957870    Tháng 6

=== Doanh thu theo nhóm SP x tháng ===
month                   4             5             6
group_code                                           
CITYBIKE_P   3.805649e+09  3.819585e+09  4.774481e+09
KIDBIKE_1    9.647336e+08  9.583232e+08  1.197904e+09
KIDBIKE_2    2.490918e+08  2.490918e+08  3.113647e+08
SPORTBIKE_A  3.723732e+08  3.723732e+08  4.654665e+08
SPORTBIKE_S  5.271280e+08  5.271280e+08  6.589100e+08


In [8]:
top20 = q2_preds.groupby(['product_code','product_name','group_code','color_std','line_name']).agg(
    pred_quantity=('quantity','sum'),
    pred_revenue=('revenue','sum')
).reset_index().nlargest(20, 'pred_quantity')
top20['rank'] = range(1, 21)
print('=== Top 20 SKU dự kiến bán chạy Q2/2026 ===')
print(top20[['rank','product_name','group_code','color_std','pred_quantity','pred_revenue']].to_string(index=False))

=== Top 20 SKU dự kiến bán chạy Q2/2026 ===
 rank                                   product_name group_code color_std  pred_quantity  pred_revenue
    1                Xe đạp Thống Nhất Puppy 20 Hồng  KIDBIKE_1      hồng     912.211146  1.195503e+09
    2                    Xe đạp Thống Nhất LD 26 Kem CITYBIKE_P       kem     885.214490  1.639884e+09
    3                   Xe đạp Thống Nhất New 24 Kem CITYBIKE_P       kem     845.836436  1.315276e+09
    4                   Xe đạp Thống Nhất New 26 Kem CITYBIKE_P       kem     646.072591  1.311937e+09
    5            Xe đạp Thống Nhất LD 24-01_2023 Kem CITYBIKE_P       kem     590.915517  9.947076e+08
    6              Xe đạp Thống Nhất New 26 Café/nâu CITYBIKE_P  café/nâu     556.388909  8.991862e+08
    7             Xe đạp Thống Nhất GN 06-26 2.0 Đen CITYBIKE_P       đen     550.717634  8.765585e+08
    8            Xe đạp Thống Nhất GN 06 20 2024 Đen  KIDBIKE_1       đen     322.472042  3.772623e+08
    9             Xe đạp Thốn

In [9]:
comparison = pd.DataFrame({
    'Model': ['Prophet', 'LightGBM'],
    'Val MAPE': [val_mape, lgb_mape],
    'Val RMSE': [val_rmse, lgb_rmse],
})
print('=== So sánh model ===')
print(comparison)

rev_fc = q2_preds.groupby('week_start').agg(revenue_lgb=('revenue','sum')).reset_index()
rev_fc.columns = ['week','revenue_lgb']
rev_fc = rev_fc.merge(q2_prophet[['week','revenue_prophet']], on='week', how='outer')
rev_fc.to_parquet(FCAST / 'q1_revenue_forecast.parquet', index=False)
top20.to_parquet(FCAST / 'q1_top20_skus.parquet', index=False)

group_rev.to_parquet(FCAST / 'q1_group_forecast.parquet', index=False)
monthly_rev.to_parquet(FCAST / 'q1_monthly_forecast.parquet', index=False)

comparison.to_parquet(FCAST / 'q1_model_comparison.parquet', index=False)
print('Q1 outputs saved.')

=== So sánh model ===
      Model  Val MAPE      Val RMSE
0   Prophet  5.515617  8.140071e+09
1  LightGBM  1.216386  2.857819e+01
Q1 outputs saved.


## Q2 - Dự báo màu sắc

In [10]:
cm = pd.read_parquet(DATA / 'color_month_panel.parquet')

last_shares = cm[cm['ym_str'] == cm['ym_str'].max()].copy()

color_fc_rows = []
for _, row in last_shares.iterrows():
    for m in [4, 5, 6]:
        delta_months = m - row['month_num']
        new_share = max(0, min(1, row['qty_share'] + row['qty_share_trend'] * delta_months))
        color_fc_rows.append({
            'color_std': row['color_std'],
            'group_code': row['group_code'],
            'month': m,
            'predicted_qty_share': new_share,
            'trend_slope': row['qty_share_trend']
        })

color_fc = pd.DataFrame(color_fc_rows)

totals_by_gm = color_fc.groupby(['group_code','month'])['predicted_qty_share'].sum().reset_index()
totals_by_gm.columns = ['group_code','month','total_share']
color_fc = color_fc.merge(totals_by_gm, on=['group_code','month'])
color_fc['predicted_qty_share'] = color_fc['predicted_qty_share'] / color_fc['total_share'].clip(lower=0.01)
color_fc = color_fc.drop(columns=['total_share'])

print(f'Color forecast: {color_fc.shape}')
print(color_fc.head(20))

Color forecast: (192, 5)
              color_std   group_code  month  predicted_qty_share  trend_slope
0            0219-05-26   CITYBIKE_P      4             0.003592    -0.001997
1            0219-05-26   CITYBIKE_P      5             0.001525    -0.001997
2            0219-05-26   CITYBIKE_P      6             0.000000    -0.001997
3                 05-26   CITYBIKE_P      4             0.027841     0.000871
4                 05-26   CITYBIKE_P      5             0.026973     0.000871
5                 05-26   CITYBIKE_P      6             0.026196     0.000871
6                 05-27   CITYBIKE_P      4             0.008582     0.000166
7                 05-27   CITYBIKE_P      5             0.008219     0.000166
8                 05-27   CITYBIKE_P      6             0.007895     0.000166
9   2026-04-16 00:00:00    KIDBIKE_1      4             0.001073     0.000000
10  2026-04-16 00:00:00    KIDBIKE_1      5             0.000977     0.000000
11  2026-04-16 00:00:00    KIDBIKE_1   

In [11]:
trend_summary = cm.groupby(['color_std','group_code']).agg(
    qty_share_trend=('qty_share_trend','first'),
    rev_share_trend=('rev_share_trend','first'),
    avg_qty_share=('qty_share','mean')
).reset_index()

rising = trend_summary[trend_summary['qty_share_trend'] > 0.005].sort_values('qty_share_trend', ascending=False)
declining = trend_summary[trend_summary['qty_share_trend'] < -0.005].sort_values('qty_share_trend')

print('=== Màu tăng nhu cầu ===')
print(rising[['color_std','group_code','qty_share_trend','avg_qty_share']].head(10).to_string(index=False))
print('\n=== Màu giảm nhu cầu ===')
print(declining[['color_std','group_code','qty_share_trend','avg_qty_share']].head(10).to_string(index=False))

=== Màu tăng nhu cầu ===
color_std  group_code  qty_share_trend  avg_qty_share
      ghi SPORTBIKE_A         0.132071       0.491724
      đen SPORTBIKE_S         0.073916       0.341226
      kem  CITYBIKE_P         0.040152       0.214375
     hồng   KIDBIKE_1         0.027491       0.219201
       đỏ SPORTBIKE_S         0.025682       0.112456
     xanh   KIDBIKE_1         0.014989       0.110020
      đen   KIDBIKE_1         0.014917       0.136306
      ghi   KIDBIKE_1         0.013832       0.067471
      kem   KIDBIKE_1         0.010073       0.095427
      kem   KIDBIKE_2         0.008567       0.058533

=== Màu giảm nhu cầu ===
 color_std  group_code  qty_share_trend  avg_qty_share
xanh dương   KIDBIKE_1        -0.203730       0.214301
       đen SPORTBIKE_A        -0.086360       0.347873
   tem cam SPORTBIKE_S        -0.070011       0.081314
      trời SPORTBIKE_A        -0.056802       0.196712
      xanh SPORTBIKE_A        -0.054370       0.075876
       tím   KIDBIKE_2   

In [12]:
sku_weekly = fs.copy()
sku_weekly['week_start'] = sku_weekly['order_date'].dt.to_period('W').apply(lambda x: x.start_time)
sku_w = sku_weekly.groupby(['product_code','product_name','color_std','group_code','week_start'])['quantity'].sum().reset_index()

max_wk = sku_w['week_start'].max()
recent_4w = sku_w[sku_w['week_start'] > max_wk - pd.Timedelta(weeks=4)]
prior_8w = sku_w[(sku_w['week_start'] > max_wk - pd.Timedelta(weeks=12)) & (sku_w['week_start'] <= max_wk - pd.Timedelta(weeks=4))]

recent_avg = recent_4w.groupby('product_code')['quantity'].mean().reset_index(name='avg_recent_4w')
prior_avg = prior_8w.groupby('product_code')['quantity'].mean().reset_index(name='avg_prior_8w')

slow = recent_avg.merge(prior_avg, on='product_code', how='outer').fillna(0)
slow['velocity_ratio'] = np.where(slow['avg_prior_8w'] > 0, slow['avg_recent_4w'] / slow['avg_prior_8w'], 1.0)

sku_info = fs.groupby('product_code').agg(
    product_name=('product_name','first'),
    color_std=('color_std','first'),
    group_code=('group_code','first')
).reset_index()
slow = slow.merge(sku_info, on='product_code', how='left')

slow['status'] = 'Bình thường'
slow.loc[(slow['velocity_ratio'] < 0.5) & (slow['avg_recent_4w'] < 2), 'status'] = 'Bán chậm'
slow.loc[(slow['velocity_ratio'] < 0.7) & (slow['velocity_ratio'] >= 0.5), 'status'] = 'Giảm'

print('=== SKU status ===')
print(slow['status'].value_counts())

slow_moving = slow[slow['status'].isin(['Bán chậm','Giảm'])].sort_values('velocity_ratio')
print(f'\nSlow-moving SKUs: {len(slow_moving)}')
print(slow_moving[['product_name','color_std','group_code','avg_recent_4w','avg_prior_8w','velocity_ratio','status']].head(20).to_string(index=False))

=== SKU status ===
status
Bình thường    114
Bán chậm        11
Giảm             9
Name: count, dtype: int64

Slow-moving SKUs: 20
                                     product_name  color_std  group_code  avg_recent_4w  avg_prior_8w  velocity_ratio   status
              Xe đạp Thống Nhất Neo 20-02 Đỏ tươi    đỏ tươi   KIDBIKE_1       0.000000      6.000000        0.000000 Bán chậm
        Xe đạp Thống Nhất MTB 26-05_2023 Xanh rêu        rêu SPORTBIKE_S       0.000000      3.750000        0.000000 Bán chậm
          Xe đạp Thống Nhất New 26 Café/nâu DA HP         hp  CITYBIKE_P       0.000000     74.000000        0.000000 Bán chậm
            Xe đạp Thống Nhất New 26 màu đỏ DA HP         hp  CITYBIKE_P       0.000000     74.000000        0.000000 Bán chậm
             Xe đạp Thống Nhất New 26 Trắng DA HP         hp  CITYBIKE_P       0.000000     76.000000        0.000000 Bán chậm
           Xe đạp Thống Nhất GN 06-26 2.0 Pro Ghi        ghi SPORTBIKE_S       0.000000      1.000000      

In [13]:
color_fc.to_parquet(FCAST / 'q2_color_forecast.parquet', index=False)
slow.to_parquet(FCAST / 'q2_slow_moving.parquet', index=False)
print('Q2 outputs saved.')

Q2 outputs saved.


## Q3 - Dự báo hoạt động đại lý

In [14]:
cust = pd.read_parquet(DATA / 'customer_features.parquet')
print(f'Customer features: {cust.shape}')
print(f'Label distribution:\n{cust["ordered_30d"].value_counts()}')

feat_cols = [
    'recency','tenure','total_orders','total_revenue','total_quantity',
    'n_skus','n_groups','n_lines','avg_unit_price','avg_order_value',
    'avg_qty_per_order','orders_per_month','revenue_per_month',
    'orders_recent','rev_recent','orders_prior','rev_prior',
    'order_trend','revenue_trend','rev_slope','qty_slope','region_enc'
]

X = cust[feat_cols].fillna(0)
y = cust['ordered_30d']

print(f'Features: {X.shape}, Positive rate: {y.mean():.3f}')

Customer features: (678, 25)
Label distribution:
ordered_30d
0    403
1    275
Name: count, dtype: int64
Features: (678, 22), Positive rate: 0.406


In [15]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs, f1s = [], []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    clf = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
        eval_metric='auc', random_state=42, verbosity=0
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    proba = clf.predict_proba(X_va)[:, 1]
    pred = (proba >= 0.5).astype(int)
    aucs.append(roc_auc_score(y_va, proba))
    f1s.append(f1_score(y_va, pred))
    print(f'Fold {fold}: AUC={aucs[-1]:.4f}, F1={f1s[-1]:.4f}')

print(f'\nMean AUC: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}')
print(f'Mean F1:  {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}')

Fold 0: AUC=0.8759, F1=0.7966


Fold 1: AUC=0.8819, F1=0.7664


Fold 2: AUC=0.9295, F1=0.8205


Fold 3: AUC=0.8957, F1=0.8000


Fold 4: AUC=0.9334, F1=0.8393

Mean AUC: 0.9033 +/- 0.0239
Mean F1:  0.8046 +/- 0.0245


In [16]:
final_clf = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    eval_metric='auc', random_state=42, verbosity=0
)
final_clf.fit(X, y)

cust['prob_order_30d'] = final_clf.predict_proba(X)[:, 1]
print(f'Probability stats:\n{cust["prob_order_30d"].describe()}')

Probability stats:
count    678.000000
mean       0.413538
std        0.454423
min        0.000562
25%        0.005612
50%        0.093916
75%        0.962476
max        0.998596
Name: prob_order_30d, dtype: float64


In [17]:
def rfm_score(series, reverse=False):
    labels = [5,4,3,2,1] if reverse else [1,2,3,4,5]
    return pd.qcut(series.rank(method='first'), q=5, labels=labels).astype(int)

cust['R_score'] = rfm_score(cust['recency'], reverse=True)
cust['F_score'] = rfm_score(cust['total_orders'])
cust['M_score'] = rfm_score(cust['total_revenue'])
cust['RFM_score'] = cust['R_score'] + cust['F_score'] + cust['M_score']

def rfm_segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal'
    elif row['R_score'] >= 3 and row['F_score'] < 3:
        return 'Potential'
    elif row['R_score'] < 3 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] < 2:
        return 'Lost'
    else:
        return 'Hibernating'

cust['rfm_segment'] = cust.apply(rfm_segment, axis=1)
print('=== RFM Segments ===')
print(cust['rfm_segment'].value_counts())

=== RFM Segments ===
rfm_segment
Champions      170
Loyal          141
Lost           112
At Risk         96
Potential       96
Hibernating     63
Name: count, dtype: int64


In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
trend_feats = cust[['orders_per_month','revenue_per_month','rev_slope']].fillna(0)
normed = scaler.fit_transform(trend_feats)
recency_score = 1 - scaler.fit_transform(cust[['recency']])

cust['trend_score'] = (
    0.3 * normed[:, 0] +
    0.3 * normed[:, 1] +
    0.2 * normed[:, 2] +
    0.2 * recency_score.flatten()
) * 100

prob_med = cust['prob_order_30d'].median()
trend_med = cust['trend_score'].median()

def marketing_priority(row):
    high_p = row['prob_order_30d'] >= prob_med
    pos_trend = row['trend_score'] >= trend_med
    if high_p and pos_trend:
        return 'Giữ chân'
    elif high_p and not pos_trend:
        return 'Cảnh báo'
    elif not high_p and pos_trend:
        return 'Phát triển'
    else:
        return 'Nguy cơ'

cust['marketing_priority'] = cust.apply(marketing_priority, axis=1)
print('=== Marketing Priority ===')
print(cust['marketing_priority'].value_counts())

=== Marketing Priority ===
marketing_priority
Nguy cơ       296
Giữ chân      296
Cảnh báo       43
Phát triển     43
Name: count, dtype: int64


In [19]:
churn_risk = cust.nlargest(30, 'recency')[[
    'customer_code','recency','total_orders','total_revenue',
    'prob_order_30d','trend_score','rfm_segment','marketing_priority'
]].copy()
churn_risk['risk_level'] = np.where(churn_risk['prob_order_30d'] < 0.3, 'Cao',
                            np.where(churn_risk['prob_order_30d'] < 0.5, 'Trung bình', 'Thấp'))
print('=== Top 30 đại lý nguy cơ rời bỏ ===')
print(churn_risk.to_string(index=False))

=== Top 30 đại lý nguy cơ rời bỏ ===
customer_code  recency  total_orders  total_revenue  prob_order_30d  trend_score rfm_segment marketing_priority risk_level
     KH-00002      419             1        3509259        0.004164    15.954554        Lost            Nguy cơ        Cao
     KH-00003      419             1        2398148        0.005182    15.951487        Lost            Nguy cơ        Cao
     KH-00005      418             1        4425926        0.029835    16.005483        Lost            Nguy cơ        Cao
     KH-00013      415             1        2490741        0.005834    16.145311        Lost            Nguy cơ        Cao
     KH-00017      415             1       23350000        0.028592    16.203430        Lost            Nguy cơ        Cao
     KH-00020      414             1      124583332        0.015282    16.534716        Lost            Nguy cơ        Cao
     KH-00021      414             1       87037037        0.004671    16.429849        Lost          

In [20]:
dealer_out = cust[['customer_code','recency','tenure','total_orders','total_revenue',
                      'orders_per_month','revenue_per_month','order_trend','revenue_trend',
                      'rev_slope','n_skus','prob_order_30d','R_score','F_score','M_score',
                      'RFM_score','rfm_segment','trend_score','marketing_priority','region']].copy()

dealer_out.to_parquet(FCAST / 'q3_dealer_probability.parquet', index=False)
churn_risk.to_parquet(FCAST / 'q3_churn_risk.parquet', index=False)
print('Q3 outputs saved.')
print(f'\n=== All forecast files ===')
for f in sorted(FCAST.glob('*.parquet')):
    print(f'  {f.name} ({f.stat().st_size/1024:.0f} KB)')

Q3 outputs saved.

=== All forecast files ===
  q1_group_forecast.parquet (3 KB)
  q1_model_comparison.parquet (2 KB)
  q1_monthly_forecast.parquet (3 KB)
  q1_revenue_forecast.parquet (3 KB)
  q1_top20_skus.parquet (6 KB)
  q2_color_forecast.parquet (6 KB)
  q2_slow_moving.parquet (10 KB)
  q3_churn_risk.parquet (6 KB)
  q3_dealer_probability.parquet (48 KB)
